In [ ]:
!pip install torch torchvision
!pip install opencv-python matplotlib tqdm
!pip install torchreid

In [ ]:
!ls /kaggle/input/models/phongtrnnguyn/reid/pytorch/default/1
!ls /kaggle/input/models/phongtrnnguyn/reid/pytorch/default/1

In [ ]:
import os
import glob
import re
import time
import shutil
import os.path as osp
import pandas as pd
import torch
import torchreid
from torchreid.reid.data import VideoDataset, register_video_dataset

class BaseKaggleDataset_MultiTask(VideoDataset):
    def __init__(self, q_dir, g_dir, q_attr_file='', g_attr_file='', **kwargs):
        
        # Load attribute dictionaries
        self.query_attrs = self.load_attributes(q_attr_file, "Query")
        self.gallery_attrs = self.load_attributes(g_attr_file, "Gallery")

        # Read images
        query = self.process_dir(q_dir)
        gallery = self.process_dir(g_dir)
        
        if not query or not gallery:
            print(f"\n[ALERT] Failed to read image data!")
            dummy = [(['fake.jpg'], 0, 0)]
            super().__init__(dummy, dummy, dummy, **kwargs)
            return

        dummy_train = [(['fake.jpg'], 0, 0)] 
        super().__init__(dummy_train, query, gallery, **kwargs)

    def load_attributes(self, csv_path, split_name):
        attr_dict = {}
        if osp.exists(csv_path):
            df = pd.read_csv(csv_path)
            for _, row in df.iterrows():
                pid = int(row['id'])
                # Extract 15 attributes into a Tensor
                attributes = torch.tensor(row.drop('id').values.astype(float))
                attr_dict[pid] = attributes
            print(f"=> [SUCCESS] Loaded {split_name} attribute dictionary: {len(attr_dict)} IDs!")
        else:
            print(f"=> [WARNING] {split_name} CSV file not found at {csv_path}")
        return attr_dict

    def process_dir(self, dir_path):
        dataset = []
        if not osp.exists(dir_path): 
            return dataset
        pids = [d for d in os.listdir(dir_path) if osp.isdir(osp.join(dir_path, d))]
        for pid_str in pids:
            try: 
                pid = int(pid_str)
            except ValueError: 
                continue
            pid_path = osp.join(dir_path, pid_str)
            for t_str in [t for t in os.listdir(pid_path) if t.lower().startswith('tracklet_')]:
                track_path = osp.join(pid_path, t_str)
                img_paths = sorted(glob.glob(osp.join(track_path, '*.[jJ][pP][gG]')))
                if not img_paths: 
                    continue
                cam_match = re.search(r'C(\d+)', osp.basename(img_paths[0]))
                cam_id = int(cam_match.group(1)) if cam_match else 0
                
                # Attribute tensor is ready in self.query_attrs[pid] or self.gallery_attrs[pid]
                # Pass a 3-element tuple to keep the standard model engine running smoothly
                dataset.append((img_paths, pid, cam_id))
        return dataset


class Case1_Dataset(BaseKaggleDataset_MultiTask):
    def __init__(self, root='', **kwargs):
        q_dir = osp.join(root, 'query', 'query')
        g_dir = osp.join(root, 'gallery', 'gallery')
        q_csv = '/kaggle/input/datasets/phongtrnnguyn/attributes/case1_aerial_to_ground/query.csv'
        g_csv = '/kaggle/input/datasets/phongtrnnguyn/attributes/case1_aerial_to_ground/gallery.csv'
        super().__init__(q_dir, g_dir, q_attr_file=q_csv, g_attr_file=g_csv, **kwargs)

class Case2_Dataset(BaseKaggleDataset_MultiTask):
    def __init__(self, root='', **kwargs):
        q_dir = osp.join(root, 'query_case2', 'query')
        g_dir = osp.join(root, 'gallery_case2', 'gallery')
        q_csv = '/kaggle/input/datasets/phongtrnnguyn/attributes/case2_ground_to_aerial/query.csv' 
        g_csv = '/kaggle/input/datasets/phongtrnnguyn/attributes/case2_ground_to_aerial/gallery.csv'
        super().__init__(q_dir, g_dir, q_attr_file=q_csv, g_attr_file=g_csv, **kwargs)

# Register datasets
dataset_case1_name = f'dataset_case1_{int(time.time())}'
dataset_case2_name = f'dataset_case2_{int(time.time())}'
register_video_dataset(dataset_case1_name, Case1_Dataset)
register_video_dataset(dataset_case2_name, Case2_Dataset)


if __name__ == '__main__':
    SAVE_DIR_ROOT = '/kaggle/working/'
    KAGGLE_DATA_ROOT = '/kaggle/input/datasets/phongtrnnguyn/ag-vpreid' 

    models_to_test = {
        'resnet50mid': '/kaggle/input/models/phongtrnnguyn/reid/pytorch/default/2/resnet50mid_weights_final_version2.pth',
        'osnet_x1_0': '/kaggle/input/models/phongtrnnguyn/reid/pytorch/default/2/osnet_x1_0_weights_final_version2.pth'
    }

    cases_to_run = {
        'case1_aerial': dataset_case1_name,
        'case2_ground': dataset_case2_name
    }

    for case_name, dataset_reg_name in cases_to_run.items():
        print(f"\n{'#'*60}\n🚀 STARTING EVALUATION: {case_name.upper()}\n{'#'*60}")

        for m_name, weight_path in models_to_test.items():
            print(f"\n{'-'*40}\n=> Running model [{m_name}] for [{case_name}]...\n{'-'*40}")
            
            datamanager = torchreid.data.VideoDataManager(
                root=KAGGLE_DATA_ROOT,
                sources=dataset_reg_name, targets=dataset_reg_name,
                height=256, width=128, batch_size_test=32, seq_len=4, workers=2
            )

            model = torchreid.models.build_model(
                name=m_name, num_classes=1, loss='softmax', pretrained=False 
            )

            try:
                torchreid.utils.load_pretrained_weights(model, weight_path)
            except Exception as e:
                print(f"=> [WEIGHTS LOAD ERROR]: File not found at {weight_path}")
                continue
                
            if torch.cuda.is_available(): 
                model = model.cuda()

            optimizer = torchreid.optim.build_optimizer(model, optim='adam', lr=0.0003)
            engine = torchreid.engine.VideoSoftmaxEngine(datamanager, model, optimizer=optimizer)

            save_dir = osp.join(SAVE_DIR_ROOT, case_name, m_name)
            engine.run(save_dir=save_dir, test_only=True, visrank=False)

            del model, engine, datamanager, optimizer
            if torch.cuda.is_available(): 
                torch.cuda.empty_cache()

    print("\n=> Compressing all result files into a ZIP file...")
    shutil.make_archive('/kaggle/working/All_Test_Results', 'zip', SAVE_DIR_ROOT)
    print(">>> 🎉 COMPLETED! Please refresh the Output section and download 'All_Test_Results.zip'.")